# Паттерн 12: Voting — параллельное голосование независимых агентов

Voting — статистический подход к надёжности. Несколько агентов независимо решают одну и ту же задачу, результат определяется большинством голосов. В LangGraph независимость обеспечивается через `Send()` — каждый агент получает собственную копию входных данных и не видит ответы остальных. Это снижает вероятность систематической ошибки.

Сценарий: классификация тональности текста. Три агента с разными промптами независимо определяют тональность (positive / negative / neutral). Агрегатор подсчитывает голоса и выбирает ответ большинства.

```mermaid
graph LR
    START((START)) -->|Send| voter1(["🔹 Voter 1<br/><i>голосует за вариант</i>"])
    START -->|Send| voter2(["🔹 Voter 2<br/><i>голосует за вариант</i>"])
    START -->|Send| voter3(["🔹 Voter 3<br/><i>голосует за вариант</i>"])
    voter1 --> aggregator(["📊 Aggregator<br/><i>подсчитывает голоса</i>"])
    voter2 --> aggregator
    voter3 --> aggregator
    aggregator --> END((END))

    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef aggregator fill:#F59E0B,stroke:#D97706,color:#fff,stroke-width:2px

    class START,END terminal
    class voter1,voter2,voter3 worker
    class aggregator aggregator
```

In [1]:
import sys
sys.path.insert(0, "..")

import operator
from typing import TypedDict, Annotated
from collections import Counter

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Состояние голосования и голосующего

Как и в Map-Reduce, для fan-out через `Send()` нужны два `TypedDict`. Основное состояние `VotingState` хранит текст, список голосов (с редьюсером `operator.add` для автоматического объединения) и итоговый вердикт. Отдельное состояние `VoterState` описывает вход конкретного голосующего: текст, его номер и персональный промпт.

In [3]:
llm = get_llm()

# --- Состояние отдельного голосующего ---
class VoterState(TypedDict):
    text: str
    voter_id: int
    prompt_style: str

# --- Основное состояние ---
class VotingState(TypedDict):
    text: str
    votes: Annotated[list[str], operator.add]
    final_verdict: str

## Промпты голосующих и fan-out через Send

Для эффективного голосования каждый агент получает уникальный промпт — это увеличивает разнообразие ответов и снижает систематическую ошибку. Функция `create_voters` создаёт список `Send()`, каждый из которых запускает узел `"voter"` с собственным `VoterState`. LangGraph выполняет все `Send()` параллельно.

In [4]:
# Разные промпты для разнообразия (снижает систематическую ошибку)
VOTER_PROMPTS = [
    "Ты — аналитик тональности. Определи эмоциональную окраску текста. Ответь ОДНИМ словом: positive, negative или neutral.",
    "Ты — лингвист. Оцени общий тон высказывания. Ответь СТРОГО одним словом: positive, negative или neutral.",
    "Ты — эксперт по коммуникации. Какой тон у этого сообщения? Одно слово: positive, negative или neutral.",
]

# --- Fan-out: создаём параллельных голосующих ---
def create_voters(state: VotingState) -> list[Send]:
    sends = []
    for i, prompt in enumerate(VOTER_PROMPTS):
        sends.append(Send("voter", {
            "text": state["text"],
            "voter_id": i + 1,
            "prompt_style": prompt,
        }))
    print(f"  [Диспетчер] Запускаю {len(sends)} независимых голосующих")
    return sends

## Голосующий агент

Один голосующий агент получает `VoterState`, вызывает LLM с персональным промптом и возвращает голос. Ключевое: агенты не видят ответы друг друга — это обеспечивает независимость. Нормализация ответа (`positive` / `negative` / `neutral`) нужна, потому что LLM может вернуть слово в произвольном формате.

In [5]:
# --- Голосующий агент ---
def voter_node(state: VoterState) -> dict:
    response = llm.invoke([
        SystemMessage(content=state["prompt_style"]),
        HumanMessage(content=state["text"])
    ])
    vote = response.content.strip().lower()
    # Нормализация ответа
    if "positive" in vote or "позитив" in vote:
        vote = "positive"
    elif "negative" in vote or "негатив" in vote:
        vote = "negative"
    else:
        vote = "neutral"

    print(f"  [Voter {state['voter_id']}] Голос: {vote}")
    return {"votes": [vote]}

## Агрегатор голосов

Агрегатор подсчитывает голоса через `Counter` и определяет победителя по большинству. Все голоса автоматически собираются в `votes` благодаря редьюсеру `operator.add` — это fan-in, встроенный в механику состояния LangGraph.

In [6]:
# --- Агрегатор: подсчёт голосов ---
def aggregator(state: VotingState) -> dict:
    counts = Counter(state["votes"])
    winner = counts.most_common(1)[0]
    print(f"  [Агрегатор] Голоса: {dict(counts)} → Победитель: {winner[0]} ({winner[1]}/{len(state['votes'])})")
    return {"final_verdict": winner[0]}

## Сборка графа

Граф использует `add_conditional_edges(START, create_voters)` для fan-out: функция `create_voters` возвращает список `Send()`, и LangGraph создаёт параллельные экземпляры узла `"voter"`. После завершения всех голосующих результаты собираются в `aggregator` (fan-in), который определяет итоговый вердикт.

In [7]:
# --- Сборка графа ---
graph = StateGraph(VotingState)

graph.add_node("voter", voter_node)
graph.add_node("aggregator", aggregator)

graph.add_conditional_edges(START, create_voters)
graph.add_edge("voter", "aggregator")
graph.add_edge("aggregator", END)

app = graph.compile()

## Запуск

Проверяем голосование на трёх текстах с разной тональностью: явно позитивном, явно негативном и нейтральном. Для каждого текста три параллельных вызова LLM через `Send()` + один вызов агрегатора.

In [8]:
texts = [
    "Этот продукт превзошёл все мои ожидания! Рекомендую каждому.",
    "Ужасный сервис. Два дня не могу решить простейшую проблему.",
    "Доставка заняла 3 дня. Упаковка стандартная. Товар соответствует описанию.",
]

for text in texts:
    print(f"\n{'=' * 60}")
    print(f"  Текст: «{text[:60]}...»")
    result = app.invoke({"text": text, "votes": [], "final_verdict": ""})
    print(f"  Вердикт: {result['final_verdict']}  (голоса: {result['votes']})")


  Текст: «Этот продукт превзошёл все мои ожидания! Рекомендую каждому....»
  [Диспетчер] Запускаю 3 независимых голосующих


  [Voter 3] Голос: positive
  [Voter 2] Голос: positive
  [Voter 1] Голос: positive
  [Агрегатор] Голоса: {'positive': 3} → Победитель: positive (3/3)
  Вердикт: positive  (голоса: ['positive', 'positive', 'positive'])

  Текст: «Ужасный сервис. Два дня не могу решить простейшую проблему....»
  [Диспетчер] Запускаю 3 независимых голосующих


  [Voter 3] Голос: negative
  [Voter 1] Голос: negative
  [Voter 2] Голос: negative
  [Агрегатор] Голоса: {'negative': 3} → Победитель: negative (3/3)
  Вердикт: negative  (голоса: ['negative', 'negative', 'negative'])

  Текст: «Доставка заняла 3 дня. Упаковка стандартная. Товар соответст...»
  [Диспетчер] Запускаю 3 независимых голосующих


  [Voter 2] Голос: neutral
  [Voter 3] Голос: neutral
  [Voter 1] Голос: neutral
  [Агрегатор] Голоса: {'neutral': 3} → Победитель: neutral (3/3)
  Вердикт: neutral  (голоса: ['neutral', 'neutral', 'neutral'])


## Итог

Voting реализуется в LangGraph через три компонента:
- **Fan-out** — `Send()` из `START` создаёт параллельных голосующих с разными промптами
- **Голосующие** — каждый агент независимо классифицирует текст (не видит ответы остальных)
- **Агрегатор** — `Counter.most_common()` определяет победителя по большинству

Уверенность можно оценить по доле голосов: 3/3 (100%) — высокая, 2/3 (67%) — средняя (можно запросить дополнительные голоса). Разные промпты для голосующих снижают систематическую ошибку и увеличивают надёжность системы.